# 🧪 시나리오 4: 접근 제어 & 보안 검증

## 시나리오
기업 보안팀이 AI Gateway에 대해 다음을 요구합니다:

| # | 보안 요구사항 | APIM 기능 | 기대 결과 |
|---|---|---|---|
| 1 | 인증 없는 요청 차단 | Subscription Key 필수 | 401 |
| 2 | 잘못된 키 차단 | 키 검증 | 401 |
| 3 | 허가된 요청 허용 | 정상 키 | 200 |
| 4 | 존재하지 않는 모델 차단 | 백엔드 라우팅 | 404 |
| 5 | 특정 IP만 허용 | ip-filter 정책 | 403 |

각 셀을 실행하면서 응답 코드와 에러 메시지를 직접 확인하세요.

In [ ]:
import os, time, json, subprocess
import requests
from dotenv import load_dotenv

# .env에서 환경 변수 자동 로드
load_dotenv("../../.env")

APIM_URL = os.getenv("APIM_URL")
SUBSCRIPTION_KEY = os.getenv("APIM_SUBSCRIPTION_KEY")
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME", "gpt-4.1-nano")
API_VERSION = "2025-04-01-preview"

assert APIM_URL, "❌ APIM_URL이 설정되지 않았습니다. .env를 확인하세요."
assert SUBSCRIPTION_KEY, "❌ APIM_SUBSCRIPTION_KEY가 설정되지 않았습니다. .env를 확인하세요."

# APIM 리소스 정보 (테스트 5: 정책 변경용)
APIM_NAME = os.getenv("APIM_NAME")
RESOURCE_GROUP = os.getenv("RESOURCE_GROUP")

BASE_URL = f"{APIM_URL}/openai/deployments/{DEPLOYMENT_NAME}/chat/completions"
BODY = {"messages": [{"role": "user", "content": "test"}], "max_tokens": 5}

print("✅ 환경 설정 완료")
print(f"   APIM: {APIM_URL}")
print(f"   API: {BASE_URL}")

results = {}  # 테스트 결과 수집용

---
## 테스트 1: 인증 없이 호출

Subscription Key 없이 API를 호출합니다.  
**기대:** `401 Unauthorized`

In [ ]:
print("▶ 테스트 1: Subscription Key 없이 호출\n")

resp = requests.post(
    BASE_URL,
    params={"api-version": API_VERSION},
    headers={"Content-Type": "application/json"},  # Key 없음!
    json=BODY,
    timeout=10
)

print(f"  HTTP Status: {resp.status_code}")
print(f"  응답 본문:")
try:
    print(json.dumps(resp.json(), indent=2, ensure_ascii=False))
except Exception:
    print(resp.text[:300])

results["인증 없는 호출 차단"] = resp.status_code == 401
print(f"\n  {'✅ PASS' if resp.status_code == 401 else '❌ FAIL'} - 기대: 401, 실제: {resp.status_code}")

---
## 테스트 2: 잘못된 키로 호출

의도적으로 유효하지 않은 키를 사용합니다.  
**기대:** `401 Unauthorized`

In [ ]:
print("▶ 테스트 2: 잘못된 Subscription Key\n")

resp = requests.post(
    BASE_URL,
    params={"api-version": API_VERSION},
    headers={
        "Content-Type": "application/json",
        "Ocp-Apim-Subscription-Key": "fake-key-this-should-not-work-12345"
    },
    json=BODY,
    timeout=10
)

print(f"  HTTP Status: {resp.status_code}")
print(f"  응답 본문:")
try:
    print(json.dumps(resp.json(), indent=2, ensure_ascii=False))
except Exception:
    print(resp.text[:300])

results["잘못된 키 차단"] = resp.status_code == 401
print(f"\n  {'✅ PASS' if resp.status_code == 401 else '❌ FAIL'} - 기대: 401, 실제: {resp.status_code}")

---
## 테스트 3: 정상 키로 호출

올바른 Subscription Key로 호출합니다.  
**기대:** `200 OK`

In [ ]:
print("▶ 테스트 3: 정상 Subscription Key\n")

resp = requests.post(
    BASE_URL,
    params={"api-version": API_VERSION},
    headers={
        "Content-Type": "application/json",
        "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY
    },
    json={"messages": [{"role": "user", "content": "Say hello in Korean"}], "max_tokens": 20},
    timeout=30
)

print(f"  HTTP Status: {resp.status_code}")
if resp.status_code == 200:
    data = resp.json()
    print(f"  모델 응답: {data['choices'][0]['message']['content']}")
    print(f"  사용 토큰: {data['usage']}")
else:
    print(f"  응답: {resp.text[:200]}")

results["정상 인증 통과"] = resp.status_code == 200
print(f"\n  {'✅ PASS' if resp.status_code == 200 else '❌ FAIL'} - 기대: 200, 실제: {resp.status_code}")

---
## 테스트 4: 존재하지 않는 모델 호출

배포되지 않은 모델명으로 호출합니다.  
**기대:** `404 Not Found` 또는 백엔드 에러

In [ ]:
print("▶ 테스트 4: 존재하지 않는 모델 (nonexistent-model)\n")

wrong_url = f"{APIM_URL}/openai/deployments/nonexistent-model-xyz/chat/completions"

resp = requests.post(
    wrong_url,
    params={"api-version": API_VERSION},
    headers={
        "Content-Type": "application/json",
        "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY
    },
    json=BODY,
    timeout=30
)

print(f"  HTTP Status: {resp.status_code}")
print(f"  응답 본문:")
try:
    print(json.dumps(resp.json(), indent=2, ensure_ascii=False))
except:
    print(f"  {resp.text[:300]}")

results["잘못된 모델 차단"] = resp.status_code != 200
print(f"\n  {'✅ PASS' if resp.status_code != 200 else '❌ FAIL'} - 200이 아니면 정상 차단 (실제: {resp.status_code})")

---
## 테스트 5: IP 필터 정책 (자동화)

실제 `<ip-filter>` 정책을 APIM에 적용/해제하면서 차단과 허용을 검증합니다.

| 단계 | 정책 | 기대 결과 |
|------|------|----------|
| 5-A | 내 IP를 `forbid`에 추가 | **403 Forbidden** |
| 5-B | 정책 제거 (원래 정책 복원) | **200 OK** |

> `az` CLI로 정책을 자동 변경합니다. Azure에 로그인되어 있어야 합니다.

In [ ]:
print("▶ 테스트 5-A: 내 IP 차단 (ip-filter forbid)\n")

# 1. 내 공인 IP 확인
my_ip = requests.get("https://api.ipify.org", timeout=5).text
print(f"  내 공인 IP: {my_ip}")

# 2. Subscription ID 가져오기
sub_id = subprocess.run(["az", "account", "show", "--query", "id", "-o", "tsv"],
                        capture_output=True, text=True).stdout.strip()

# API 이름 확인 (환경에 따라 다를 수 있음)
API_NAME = os.getenv("APIM_API_NAME", "azure-openai-api")
POLICY_URI = f"https://management.azure.com/subscriptions/{sub_id}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.ApiManagement/service/{APIM_NAME}/apis/{API_NAME}/policies/policy?api-version=2023-09-01-preview"

# 3. 현재 정책 백업
backup_result = subprocess.run(
    ["az", "rest", "--method", "GET", "--uri", POLICY_URI],
    capture_output=True, text=True
)
if backup_result.returncode != 0:
    print(f"  ❌ 정책 백업 실패: {backup_result.stderr[:200]}")
    print(f"  → APIM_NAME, RESOURCE_GROUP, APIM_API_NAME 환경 변수를 확인하세요.")
    results["IP 차단 (forbid)"] = False
else:
    original_policy = json.loads(backup_result.stdout)["properties"]["value"]
    print(f"  ✅ 기존 정책 백업 완료 ({len(original_policy)} chars)")

    # 4. ip-filter forbid 정책 적용
    blocked_policy = original_policy.replace(
        "<base />",
        f'<base />\n        <ip-filter action="forbid">\n            <address>{my_ip}</address>\n        </ip-filter>',
        1  # inbound의 첫 번째 <base />만 교체
    )

    apply_body = json.dumps({"properties": {"format": "rawxml", "value": blocked_policy}})
    apply_result = subprocess.run(
        ["az", "rest", "--method", "PUT", "--uri", POLICY_URI,
         "--body", apply_body, "--headers", "Content-Type=application/json"],
        capture_output=True, text=True
    )
    print(f"  정책 적용: {'✅ 성공' if apply_result.returncode == 0 else '❌ 실패'}")
    if apply_result.returncode != 0:
        print(f"  에러: {apply_result.stderr[:200]}")

    # 5. 정책 반영 대기
    time.sleep(3)

    # 6. API 호출 → 403 기대
    resp = requests.post(
        BASE_URL,
        params={"api-version": API_VERSION},
        headers={"Content-Type": "application/json", "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY},
        json=BODY,
        timeout=30
    )

    print(f"\n  HTTP Status: {resp.status_code}")
    try:
        print(f"  응답: {json.dumps(resp.json(), indent=2, ensure_ascii=False)}")
    except:
        print(f"  응답: {resp.text[:300]}")

    results["IP 차단 (forbid)"] = resp.status_code == 403
    print(f"\n  {'✅ PASS' if resp.status_code == 403 else '❌ FAIL'} - 기대: 403, 실제: {resp.status_code}")

    # ⚠️ 안전장치: 테스트 성공/실패와 관계없이 즉시 정책 복원 시도
    print("\n  ⚠️ 안전장치: 정책 자동 복원 중...")
    restore_body = json.dumps({"properties": {"format": "rawxml", "value": original_policy}})
    restore_result = subprocess.run(
        ["az", "rest", "--method", "PUT", "--uri", POLICY_URI,
         "--body", restore_body, "--headers", "Content-Type=application/json"],
        capture_output=True, text=True
    )
    print(f"  정책 복원: {'✅ 성공' if restore_result.returncode == 0 else '❌ 실패 — 수동 복원 필요!'}")

In [ ]:
print("▶ 테스트 5-B: 정책 복원 확인 (정상 호출 가능한지)\n")

# 5-A에서 이미 자동 복원됨 → 정상 호출 가능한지 확인
time.sleep(3)

resp = requests.post(
    BASE_URL,
    params={"api-version": API_VERSION},
    headers={
        "Content-Type": "application/json",
        "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY
    },
    json=BODY,
    timeout=30
)

print(f"  HTTP Status: {resp.status_code}")
print(f"  Backend: {resp.headers.get('x-backend-url', 'n/a')}")
if resp.status_code == 200:
    print(f"  모델 응답: {resp.json()['choices'][0]['message']['content']}")

results["IP 차단 해제 (복원)"] = resp.status_code == 200
print(f"\n  {'✅ PASS' if resp.status_code == 200 else '❌ FAIL'} - 기대: 200, 실제: {resp.status_code}")

if resp.status_code != 200:
    print(f"\n  ⚠️ 정책 복원이 안 된 경우 수동 복원:")
    print(f"  → Azure Portal → APIM → APIs → {API_NAME} → Policies 에서 ip-filter 제거")

---
## 전체 결과 요약

In [ ]:
print("═" * 50)
print(" 접근 제어 & 보안 테스트 결과")
print("═" * 50)
print()

total = len(results)
passed = sum(1 for v in results.values() if v)

for check, ok in results.items():
    icon = "✅" if ok else "❌"
    print(f"  {icon} {check}")

print()
print(f"  결과: {passed}/{total} 통과")
print("═" * 50)